# Build a Branded Contract Package in 5 Steps

**A CompleteTech LLC walkthrough - `agentic_contract_skill`**

This notebook teaches you how to turn a single INI file into two production-grade PDFs:

1. A multi-page **Agentic Development Services Agreement** (contract).
2. A printable **#10 addressed envelope** for mailing the contract.

Everything is driven from `config.ini`. You change the data; the generator handles the formatting, branding, signature block, and envelope.

---

## Prerequisites

- Python 3.10+
- Run from the project root (the folder containing `generate_contract.py`).
- Install runtime dependencies:
  ```bash
  pip install -r requirements.txt
  ```
- For inline PDF previews in this notebook, also install **PyMuPDF**:
  ```bash
  pip install pymupdf
  ```
  (If PyMuPDF isn't installed, the preview cells will print file paths instead of rendering — generation still works.)

### Notebook helpers

Two small helpers we'll reuse throughout: `run(...)` invokes the generators as a real subprocess (so what you copy-paste into a terminal works identically), and `show_pdf(...)` renders a PDF page inline.

In [ ]:
import subprocess
import sys
from pathlib import Path
from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path.cwd()
PREVIEW_DIR = PROJECT_ROOT / "preview"
PREVIEW_DIR.mkdir(exist_ok=True)

def run(*args: str) -> None:
    """Run a subprocess command, stream its output, and raise on nonzero exit."""
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run([str(a) for a in args], cwd=PROJECT_ROOT,
                            capture_output=True, text=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr, end="", file=sys.stderr)
        raise RuntimeError(f"Command exited {result.returncode}")

def show_pdf(pdf_path: str, page: int = 0, dpi: int = 110, width: int = 900) -> None:
    """Render one page of a PDF and display it inline. Falls back to a path link
    if PyMuPDF isn't installed."""
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        print(f"(missing) {pdf_path}")
        return
    try:
        import fitz  # PyMuPDF
    except ImportError:
        display(Markdown(f"_PyMuPDF not installed — PDF saved at `{pdf_path}`._"))
        return
    out_png = PREVIEW_DIR / f"{pdf_path.stem}_page-{page+1}.png"
    fitz.open(pdf_path)[page].get_pixmap(dpi=dpi).save(out_png)
    display(Image(filename=str(out_png), width=width))


### What we'll build

If you've already run the generators once, the project ships with previews you can peek at now to set expectations:

In [ ]:
for name in ["contract_page-1.png", "envelope_page-1.png"]:
    p = PREVIEW_DIR / name
    if p.exists():
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(p), width=520))

---
## Step 1 - Understand the Pipeline

One config, one generator, two outputs:

```
config.ini ───► generate_contract.py ──┬─► contract.pdf
                                      └─► envelope.pdf
```

Every section of `config.ini` maps to a real-world business detail. The generator fills in the blanks against a Jinja2 template for the contract and a dedicated ReportLab layout for the envelope. Let's see the sections that drive everything:

In [ ]:
import configparser

cfg = configparser.ConfigParser(interpolation=None)
cfg.read("config.ini", encoding="utf-8")

for section in cfg.sections():
    keys = ", ".join(cfg[section].keys())
    print(f"[{section}]\n  {keys}\n")


> **What just happened:** we read the same INI file the generator will. `[provider]` and `[branding]` define the sender and visual identity; `[client]` and `[agreement]` drive the contract; `[envelope]` adds mailing details.

---
## Step 2 - Configure Your Brand

The `[branding]` block decides the visual identity for the contract package: logo, monogram, accent colors, watermark, and page furniture. Change one value here and both PDFs update on the next run.

In [ ]:
print("Primary logo (used on contract cover and letterhead):")
display(Image(filename="assets/logo.png", width=320))

In [ ]:
branding = cfg["branding"]
swatches = {
    "accent":      branding.get("accent_color", "#1E3A8A"),
    "accent_dark": branding.get("accent_color_dark", "#0F172A"),
    "accent_soft": branding.get("accent_color_soft", "#EEF2FF"),
    "muted":       branding.get("muted_color", "#64748B"),
}

html = '<div style="display:flex;gap:12px;font-family:sans-serif">' + "".join(
    f'<div style="text-align:center">'
    f'<div style="width:90px;height:90px;background:{hexv};border:1px solid #ccc;border-radius:6px"></div>'
    f'<div style="font-size:11px;margin-top:4px"><b>{name}</b><br>{hexv}</div></div>'
    for name, hexv in swatches.items()
) + '</div>'
display(Markdown(html))


> **Try this:** edit `accent_color` in `config.ini` and re-run Step 3. Accent rules, cover details, and envelope styling pick up the new color automatically.

---
## Step 3 — Generate the Contract

The contract is rendered from `templates/agentic_development_agreement.md.j2` — a Jinja2 Markdown template — and compiled to a multi-page PDF with cover page, letterhead band, watermark, footer, and signature block.

We'll run with `--no-envelope` so this step produces **only** the contract; we'll cover the envelope in Step 4.

In [ ]:
run("python", "generate_contract.py",
    "--config", "config.ini",
    "--out", "output/notebook_contract.pdf",
    "--no-envelope")


In [ ]:
show_pdf("output/notebook_contract.pdf", page=0)  # cover page


In [ ]:
show_pdf("output/notebook_contract.pdf", page=1)  # first content page with letterhead band


> **What just happened:** Jinja2 substituted every `{{ provider.legal_name }}`-style placeholder with values from `config.ini`, the resulting Markdown was converted to ReportLab flowables, and the page-furniture callbacks drew the letterhead band, watermark, and footer.

**Extension:** stack INIs to customize the contract per real client — `python generate_contract.py --config config.ini examples/minimum_client_override.ini --out output/acme_contract.pdf`.

---
## Step 4 — Generate the Addressed Envelope

The envelope is its own **#10-sized** PDF (9.5" × 4.125"), with the return address pulled from `[provider]` and the recipient block from `[client]`. Printers typically handle envelopes on a separate tray, so it's a separate file by design.

Re-running the contract generator (without `--no-envelope`) produces both PDFs — we point `--envelope-out` at a notebook-specific filename so we don't collide with the existing demo output.

In [ ]:
run("python", "generate_contract.py",
    "--config", "config.ini",
    "--out", "output/notebook_contract.pdf",
    "--envelope-out", "output/notebook_envelope.pdf")


In [ ]:
show_pdf("output/notebook_envelope.pdf", page=0)


> **What just happened:** the same `generate_contract.py` ran end-to-end, but this time the envelope sub-routine fired and wrote a second PDF. The return address strips the company name (it's already in the logo banner), the recipient block prepends `recipient_attention` from `[envelope]`, and the watermark mirrors the contract for consistency.

**Try this:** set `envelope_enabled = no` under `[branding]` to skip envelope creation entirely — useful for clients who don't need a mailed contract.

---
## Step 5 - Customize, Re-Run, Extend

Two patterns for taking this beyond the demo. Pick whichever fits your workflow - they're not mutually exclusive.

### Pattern 1 - Per-client contract overrides

Stack INIs; later files win. Keep your base brand in `config.ini` and write a tiny override per client:

```bash
python generate_contract.py   --config config.ini examples/minimum_client_override.ini   --out output/acme_contract.pdf   --envelope-out output/acme_envelope.pdf
```

### Pattern 2 - Agent integration

An agent skill invokes the generator the same way you just did. A minimal skill action looks like:

```python
import subprocess


def issue_contract(config_paths: list[str], out_path: str) -> str:
    subprocess.run(
        ["python", "generate_contract.py", "--config", *config_paths, "--out", out_path],
        check=True,
    )
    return out_path
```

See `SKILL.md` -> *Agent Operating Guidance* for the full agent-side contract.

---
## Where to take this next

- **Swap the contract template** - `templates/agentic_development_agreement.md.j2` is plain Jinja2 + Markdown. Drop in your own clauses, add jurisdiction-specific language, or fork the file per practice area.
- **Add companion contract artifacts** - statement of work, NDA, invoice, or change order templates can reuse the same config and branding helpers.
- **Wire into a CRM** - pull client rows from your CRM and generate one contract package per engagement.
- **Schedule it** - use the project's cron/scheduling skill to issue monthly retainer contracts automatically.

Demonstration disclaimer: the contract template is illustrative, not legal advice - replace with your counsel-reviewed terms before any real engagement.